# Cleora for Recommendation

## 1. Introduction
Cleora is a tool for producing dense vector representations of entities (embeddings) from a collection of their discrete connections. It is a graph embedding method that can be used to generate embeddings for users and items in a recommendation system. The embeddings can then be used to calculate similarities and generate recommendations.

## 2. Imports and Installation

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
import pandas as pd
import numpy as np
from pycleora import SparseMatrix
import json
from sklearn.metrics.pairwise import cosine_similarity
from helpers.data_loaders import load_movielens_data, load_steam_data
import itertools

## 3. Data Loading and Preparation

In [2]:
movies_df, ratings_df = load_movielens_data()
reviews_df_steam, items_df_steam = load_steam_data()

ratings_df_filtered = ratings_df[ratings_df['rating'] >= 4.0].copy()
movielens_interactions = pd.DataFrame({
    'user_id': 'movielens_user_' + ratings_df_filtered['userId'].astype(str),
    'item_id': 'movielens_item_' + ratings_df_filtered['movieId'].astype(str)
})

steam_interactions = pd.DataFrame({
    'user_id': 'steam_user_' + reviews_df_steam['user_id'].astype(str),
    'item_id': 'steam_item_' + reviews_df_steam['app_id'].astype(str)
})

all_interactions = pd.concat([movielens_interactions, steam_interactions]).drop_duplicates()

print(f"Total unique interactions: {len(all_interactions)}")

user_items = all_interactions.groupby('user_id')['item_id'].apply(list).values

item_users = all_interactions.groupby('item_id')['user_id'].apply(list).values

cleora_input_user_items = map(lambda x: ' '.join(x), user_items)
cleora_input_item_users = map(lambda x: ' '.join(x), item_users)
cleora_input = itertools.chain(cleora_input_user_items, cleora_input_item_users)

Total unique interactions: 12497401


## 4. Training and Loading Embeddings

In [3]:
mat = SparseMatrix.from_iterator(cleora_input, columns='complex::reflexive::entity')

embeddings_matrix = mat.initialize_deterministically(128)

# Perform Markov random walk
NUM_WALKS = 5
for i in range(NUM_WALKS):
    embeddings_matrix = mat.left_markov_propagate(embeddings_matrix)
    embeddings_matrix /= np.linalg.norm(embeddings_matrix, ord=2, axis=-1, keepdims=True)

embeddings = {entity: embedding for entity, embedding in zip(mat.entity_ids, embeddings_matrix)}
print(f"pycleora training complete. Loaded {len(embeddings)} embeddings.")

pycleora training complete. Loaded 228079 embeddings.


## 5. Making Recommendations

In [4]:
def get_recommendations_cleora(user_id_str, embeddings, top_k=10):
    if user_id_str not in embeddings:
        print(f"User '{user_id_str}' not found in embeddings.")
        return
    
    user_embedding = embeddings[user_id_str].reshape(1, -1)
    
    item_embeddings = {item_id: emb for item_id, emb in embeddings.items() if '_item_' in item_id}
    item_ids = list(item_embeddings.keys())
    item_matrix = np.array(list(item_embeddings.values()))
    
    similarities = cosine_similarity(user_embedding, item_matrix).flatten()
    
    top_k_indices = np.argsort(similarities)[-top_k:][::-1]
    
    print(f"Top {top_k} recommendations for user '{user_id_str}':")
    for i in top_k_indices:
        item_id_str = item_ids[i]
        score = similarities[i]
        if 'steam_item_' in item_id_str:
            item_info = items_df_steam.loc[items_df_steam['app_id'] == int(item_id_str.split('_')[-1])]
            if not item_info.empty:
                print(f"  - [steam] Item: {item_id_str}, Title: {item_info['title'].values[0]} , Score: {score:.4f}")
        elif 'movielens_item_' in item_id_str:
            item_info = movies_df.loc[movies_df['movieId'] == int(item_id_str.split('_')[-1])]
            if not item_info.empty:
                print(f"  - [movie] Item: {item_id_str}, Title: {item_info['title'].values[0]} , Score: {score:.4f}")

## 6. Example Recommendations

In [5]:
sample_user_id = 'steam_user_LydiaMorley'
get_recommendations_cleora(sample_user_id, embeddings)

Top 10 recommendations for user 'steam_user_LydiaMorley':
  - [steam] Item: steam_item_248350, Title: Omegalodon , Score: 0.2636
  - [steam] Item: steam_item_51100, Title: Tactical Intervention , Score: 0.2600
  - [steam] Item: steam_item_34870, Title: Sniper: Ghost Warrior 2 , Score: 0.2457
  - [steam] Item: steam_item_280010, Title: Gunjitsu , Score: 0.2402
  - [steam] Item: steam_item_18820, Title: Zero Gear , Score: 0.2357
  - [steam] Item: steam_item_21100, Title: F.E.A.R. 3 , Score: 0.2355
  - [steam] Item: steam_item_332250, Title: The Next Penelope , Score: 0.2303
  - [steam] Item: steam_item_277520, Title: Albedo: Eyes from Outer Space , Score: 0.2294
  - [steam] Item: steam_item_442080, Title: Riders of Icarus , Score: 0.2258
  - [steam] Item: steam_item_235980, Title: Tetrobot and Co. , Score: 0.2201


## 7. Incorporating Genres
Now, let's enrich our input by adding item-genre relationships. This will help Cleora learn embeddings that capture not just collaborative filtering patterns but also content-based similarities.

In [6]:
# Prepare item-genre relationships for MovieLens
movielens_genres = movies_df.copy()
movielens_genres['genres'] = movielens_genres['genres'].str.split('|')
movielens_genres = movielens_genres.explode('genres')
movielens_genres = movielens_genres[movielens_genres['genres'] != '(no genres listed)']
movielens_genres['item_id'] = 'movielens_item_' + movielens_genres['movieId'].astype(str)
movielens_genres['genre_id'] = 'genre_' + movielens_genres['genres'].str.lower().str.replace(' ', '_').str.replace('-', '_')
movielens_item_genre_interactions = movielens_genres[['item_id', 'genre_id']].drop_duplicates()

# Prepare item-genre relationships for Steam
steam_genres = items_df_steam.copy()
steam_genres = steam_genres.dropna(subset=['genres'])
steam_genres['genres'] = steam_genres['genres'].apply(lambda x: x if isinstance(x, list) else [])
steam_genres = steam_genres.explode('genres')
steam_genres['item_id'] = 'steam_item_' + steam_genres['app_id'].astype(str)
steam_genres['genre_id'] = 'genre_' + steam_genres['genres'].str.lower().str.replace(' ', '_').str.replace('-', '_')
steam_item_genre_interactions = steam_genres[['item_id', 'genre_id']].drop_duplicates()

# Combine all interactions
all_item_genre_interactions = pd.concat([movielens_item_genre_interactions, steam_item_genre_interactions])

# Create input for Cleora
item_genres = all_item_genre_interactions.groupby('item_id')['genre_id'].apply(list).values
genre_items = all_item_genre_interactions.groupby('genre_id')['item_id'].apply(list).values

cleora_input_item_genres = map(lambda x: ' '.join(x), item_genres)
cleora_input_genre_items = map(lambda x: ' '.join(x), genre_items)

# The full input now contains user-item, item-user, item-genre, and genre-item interactions
cleora_input_with_genres = itertools.chain(
    map(lambda x: ' '.join(x), user_items), 
    map(lambda x: ' '.join(x), item_users), 
    cleora_input_item_genres, 
    cleora_input_genre_items
)

print(f"Total unique item-genre interactions: {len(all_item_genre_interactions)}")

Total unique item-genre interactions: 178796


## 8. Training and Loading Embeddings with Genres

In [7]:
mat_with_genres = SparseMatrix.from_iterator(cleora_input_with_genres, columns='complex::reflexive::entity')
embeddings_matrix_with_genres = mat_with_genres.initialize_deterministically(128)

# Perform Markov random walk
for i in range(NUM_WALKS):
    embeddings_matrix_with_genres = mat_with_genres.left_markov_propagate(embeddings_matrix_with_genres)
    embeddings_matrix_with_genres /= np.linalg.norm(embeddings_matrix_with_genres, ord=2, axis=-1, keepdims=True)

embeddings_with_genres = {entity: embedding for entity, embedding in zip(mat_with_genres.entity_ids, embeddings_matrix_with_genres)}
print(f"pycleora training complete. Loaded {len(embeddings_with_genres)} embeddings (with genres).")

pycleora training complete. Loaded 272836 embeddings (with genres).


## 9. Example Recommendations with Genre-aware Embeddings

In [8]:
print("Recommendations without genres:")
get_recommendations_cleora(sample_user_id, embeddings)
print("\nRecommendations with genres:")
get_recommendations_cleora(sample_user_id, embeddings_with_genres)

Recommendations without genres:
Top 10 recommendations for user 'steam_user_LydiaMorley':
  - [steam] Item: steam_item_248350, Title: Omegalodon , Score: 0.2636
  - [steam] Item: steam_item_51100, Title: Tactical Intervention , Score: 0.2600
  - [steam] Item: steam_item_34870, Title: Sniper: Ghost Warrior 2 , Score: 0.2457
  - [steam] Item: steam_item_280010, Title: Gunjitsu , Score: 0.2402
  - [steam] Item: steam_item_18820, Title: Zero Gear , Score: 0.2357
  - [steam] Item: steam_item_21100, Title: F.E.A.R. 3 , Score: 0.2355
  - [steam] Item: steam_item_332250, Title: The Next Penelope , Score: 0.2303
  - [steam] Item: steam_item_277520, Title: Albedo: Eyes from Outer Space , Score: 0.2294
  - [steam] Item: steam_item_442080, Title: Riders of Icarus , Score: 0.2258
  - [steam] Item: steam_item_235980, Title: Tetrobot and Co. , Score: 0.2201

Recommendations with genres:
Top 10 recommendations for user 'steam_user_LydiaMorley':
  - [steam] Item: steam_item_51100, Title: Tactical Inter